In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"

In [ ]:
%pip install -q transformers==4.57.1 datasets accelerate bitsandbytes peft evaluate sacrebleu sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gc
import json
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

In [ ]:
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


In [ ]:
dataset = load_dataset("opus_books", "en-es")
dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

def rename_columns(example):
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["es"],
    }

dataset = dataset.map(rename_columns, remove_columns=dataset["train"].column_names)

# Debug-safe subset first
dataset["train"] = dataset["train"].select(range(8000))
dataset["test"] = dataset["test"].select(range(500))

print(dataset)
print("Train size:", len(dataset["train"]))
print("Test size:", len(dataset["test"]))
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['src', 'tgt'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['src', 'tgt'],
        num_rows: 500
    })
})
Train size: 8000
Test size: 500
{'src': '"That which has happened me in meeting you, mighty prince," replied Don Quixote, "cannot be unfortunate, even if my fall had not stopped short of the depths of the bottomless pit, for the glory of having seen you would have lifted me up and delivered me from it.', 'tgt': '-El que yo he tenido en veros, valeroso príncipe -respondió don Quijote-, es imposible ser malo, aunque mi caída no parara hasta el profundo de los abismos, pues de allí me levantara y me sacara la gloria de haberos visto.'}


In [ ]:
prefix = "translate English to Spanish: "

def build_input(src):
    return prefix + src

print(build_input("Hello, how are you?"))

translate English to Spanish: Hello, how are you?


In [ ]:
model_id = "google/flan-t5-small"

baseline_tokenizer = AutoTokenizer.from_pretrained(model_id)

baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

baseline_model.eval()

print("Baseline loaded:", model_id)

`torch_dtype` is deprecated! Use `dtype` instead!


Baseline loaded: google/flan-t5-small


In [ ]:
def generate_translation(model, tokenizer, src, max_new_tokens=80):
    text = build_input(src)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

In [ ]:
sample_text = dataset["test"][0]["src"]
reference_text = dataset["test"][0]["tgt"]

baseline_prediction = generate_translation(
    baseline_model,
    baseline_tokenizer,
    sample_text,
)

print("SOURCE:")
print(sample_text)

print("\nREFERENCE:")
print(reference_text)

print("\nBASELINE OUTPUT:")
print(baseline_prediction)

SOURCE:
Kitty, on the contrary, was more active than usual and even more animated.

REFERENCE:
Kitty, al contrario, estaba más activa a incluso más animada que nunca.

BASELINE OUTPUT:
Kitty, en el contrario, fue más activo que el habitado y incluso más animated.


In [ ]:
test_subset = dataset["test"].select(range(50))

sources = []
references = []
baseline_predictions = []

for example in test_subset:
    src = example["src"]
    tgt = example["tgt"]

    pred = generate_translation(
        baseline_model,
        baseline_tokenizer,
        src,
    )

    sources.append(src)
    references.append(tgt)
    baseline_predictions.append(pred)

print("Done generating baseline predictions.")
print("Example baseline:", baseline_predictions[0])

Done generating baseline predictions.
Example baseline: Kitty, en el contrario, fue más activo que el habitado y incluso más animated.


In [ ]:
bleu = evaluate.load("sacrebleu")

baseline_bleu = bleu.compute(
    predictions=baseline_predictions,
    references=[[ref] for ref in references],
)

print("Baseline BLEU:")
print(baseline_bleu)

Baseline BLEU:
{'score': 5.347977855587551, 'counts': [315, 92, 25, 8], 'totals': [941, 891, 841, 792], 'precisions': [33.4750265674814, 10.325476992143658, 2.972651605231867, 1.0101010101010102], 'bp': 0.9422250198633814, 'sys_len': 941, 'ref_len': 997}


In [ ]:
del baseline_model
gc.collect()
torch.cuda.empty_cache()

print("Baseline cleared.")

Baseline cleared.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("4-bit model loaded:", model_id)

4-bit model loaded: google/flan-t5-small


In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


In [ ]:
max_input_length = 128
max_target_length = 128

def preprocess_batch(examples):
    inputs = [build_input(src) for src in examples["src"]]
    targets = examples["tgt"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print(tokenized)
print(tokenized["train"][0].keys())

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("data_collator created")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan_t5_qlora_translation",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=2,
    logging_steps=20,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    max_grad_norm=0.3,
)

print("training_args created")

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Trainer initialized.")

In [ ]:
batch = data_collator([tokenized["train"][0], tokenized["train"][1]])

print(batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)
print("labels unique:", torch.unique(batch["labels"]))
print("non-masked labels:", (batch["labels"] != -100).sum())

In [ ]:
trainer.train()

In [ ]:
model.eval()

finetuned_predictions = []

for src in sources:
    pred = generate_translation(
        model,
        tokenizer,
        src,
    )
    finetuned_predictions.append(pred)

print("FINETUNED EXAMPLE:")
print(finetuned_predictions[0])

In [ ]:
finetuned_bleu = bleu.compute(
    predictions=finetuned_predictions,
    references=[[ref] for ref in references],
)

print("Fine-tuned BLEU:")
print(finetuned_bleu)

In [ ]:
for i in range(5):
    print("=" * 80)
    print("SOURCE:    ", sources[i])
    print("REFERENCE: ", references[i])
    print("BASELINE:  ", baseline_predictions[i])
    print("QLORA FT:  ", finetuned_predictions[i])

In [ ]:
model.save_pretrained("./flan_t5_qlora_adapter")
tokenizer.save_pretrained("./flan_t5_qlora_adapter")

print("QLoRA adapter and tokenizer saved.")

In [ ]:
results = {
    "model_id": model_id,
    "method": "QLoRA",
    "sources": sources,
    "references": references,
    "baseline_predictions": baseline_predictions,
    "finetuned_predictions": finetuned_predictions,
    "baseline_bleu": baseline_bleu,
    "finetuned_bleu": finetuned_bleu,
}

with open("flan_t5_qlora_translation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Results saved to flan_t5_qlora_translation_results.json")